# Task 03 — Gold Aggregation

**Workshop: Final Pipeline | Stage 3 of 3**

## Goal

Read `bronze.lab_orders_clean`, aggregate to daily revenue metrics, write the result as a managed Delta table in the Gold layer.

```
bronze.lab_orders_clean
        │
        ▼  to_date · groupBy · agg (count, sum, avg)
        │
        ▼
gold.lab_daily_orders
```

## Expected output schema

| Column | Type | Description |
|--------|------|-------------|
| order_date | DateType | Truncated from `order_datetime` |
| order_count | LongType | Number of orders per day |
| total_revenue | DoubleType | SUM of `net_amount` |
| avg_order_value | DoubleType | AVG of `net_amount`, rounded to 2 dp |

> Use `net_amount` (not `total_amount`) — it already includes discount.


In [ ]:
%run ../../setup/00_setup

In [ ]:
# ── Widgets ──
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("gold_schema",   GOLD_SCHEMA,   "Gold Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
gold_schema   = dbutils.widgets.get("gold_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders_clean"
target_table = f"{catalog}.{gold_schema}.lab_daily_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")


---

## Exercise 1 — Imports

Import `col`, `count`, `sum`, `avg`, `round`, `to_date` from `pyspark.sql.functions`.


In [ ]:
# 💡 HINT — aggregate functions:
#
#   from pyspark.sql.functions import (
#       col, to_date,
#       count, sum, avg, round
#   )
#
#   Note: Python has built-in sum() and round() — importing from
#   pyspark.sql.functions overrides them in this scope.
#   If you need Python builtins later, alias the import:
#       from pyspark.sql.functions import sum as spark_sum

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Read source table


In [ ]:
# 💡 HINT — spark.table():
#
#   df = spark.table("fully.qualified.table_name")
#
#   The table must exist before this cell runs.
#   If it doesn't: run Task 02 first.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
print(f"Source rows : {orders_clean.count():,}")
orders_clean.printSchema()

---

## Exercise 3 — Add `order_date` column

Extract the date part from `order_datetime` (TimestampType → DateType).  
Store the result in a new column called `order_date`.


In [ ]:
# 💡 HINT — to_date():
#
#   to_date(col("timestamp_column"))
#       → extracts the date part (year-month-day) from a Timestamp column
#       → returns DateType
#
#   Use withColumn to add "order_date" derived from "order_datetime".
#   You can chain this with the groupBy in Exercise 4, or do it here first.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 4 — Group and aggregate

Group by `order_date` and compute the 3 metrics from the Expected output table.  
Sort the result by `order_date` ascending.


In [ ]:
# 💡 HINT — groupBy + agg:
#
#   daily_df = (
#       df
#       .groupBy("column_to_group_by")
#       .agg(
#           count("*").alias("alias_name"),
#           sum("numeric_col").alias("alias_name"),
#           round(avg("numeric_col"), 2).alias("alias_name"),
#       )
#       .orderBy("column_to_group_by")
#   )
#
#   .agg() accepts any number of Column expressions.
#   Always give each aggregation a meaningful alias with .alias("name").

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify
print(f"Daily rows : {daily_df.count():,}  (should equal number of distinct order dates)")
daily_df.show(10, truncate=False)


---

## Exercise 5 — Write to Gold layer


In [ ]:
# 💡 HINT — write to managed Delta table:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")
#       .saveAsTable("catalog.schema.table_name")
#
#   Gold tables are typically full-refresh (overwrite) because they
#   are derived / aggregated — source of truth is always Bronze/Silver.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Final verification
row_count = spark.table(target_table).count()
print(f"✅ {target_table}  →  {row_count:,} daily records")
display(spark.table(target_table))


In [ ]:
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "table": target_table,
    "rows": row_count
}))
